In [ ]:
import openpyxl, re
import ipywidgets as widgets
from IPython.display import display

EXCEL_FILE = "plantilla_anotacion.xlsx"
LETRAS = ["A", "B", "C", "D"]

# ── Widgets ──────────────────────────────────────────────────────────────────
w_prompt  = widgets.Dropdown(options=list(range(1, 40)), description="Prompt:")
w_modelo  = widgets.Dropdown(options=["GPT-5", "Gemini 3 Flash", "Qwen 3.6 Plus"], description="Modelo:")
w_texto   = widgets.Textarea(
    placeholder="Pega aquí el resultado completo del LLM...",
    layout=widgets.Layout(width="700px", height="220px")
)
w_boton   = widgets.Button(description="Guardar", button_style="success",
                           layout=widgets.Layout(width="150px", height="40px"))
w_estado  = widgets.Output()

# ── Lógica ───────────────────────────────────────────────────────────────────
def guardar(b):
    w_estado.clear_output()
    texto = w_texto.value

    with w_estado:
        # Ranking
        m = re.search(r'Ranking[:\s]*([A-D\s>]+)', texto, re.IGNORECASE)
        if not m:
            print("❌ No encontré 'Ranking:' en el texto"); return
        letras = [x.strip().upper() for x in m.group(1).strip().split('>')]
        if len(letras) != 4 or set(letras) != set(LETRAS):
            print(f"❌ Ranking inválido: {letras}"); return
        ranking = " > ".join(letras)

        punts = {}
        for L in LETRAS:
            m2 = re.search(rf'Puntuaci[oó]n\s*{L}[:\s]*(\d+)', texto, re.IGNORECASE)
            if not m2:
                print(f"❌ No encontré 'Puntuacion {L}:'"); return
            v = int(m2.group(1))
            if not 1 <= v <= 10:
                print(f"❌ Puntuación {L}={v} fuera de 1-10"); return
            punts[L] = v

        mj = re.search(r'Justificaci[oó]n[:\s]*(.+)', texto, re.IGNORECASE | re.DOTALL)
        justificacion = mj.group(1).strip() if mj else ""

        try:
            wb = openpyxl.load_workbook(EXCEL_FILE)
            modelo_mapa = {
                "GPT-5":          "Resultados_GPT5",
                "Gemini 3 Flash": "Resultados_Gemini3Flash",
                "Qwen 3.6 Plus":  "Resultados_Qwen36Plus"
            }
            sheet_name = modelo_mapa[w_modelo.value]
            ws = wb[sheet_name]
            row = w_prompt.value + 1
            ws[f'C{row}'] = ranking
            ws[f'D{row}'] = punts['A']
            ws[f'E{row}'] = punts['B']
            ws[f'F{row}'] = punts['C']
            ws[f'G{row}'] = punts['D']
            if justificacion:
                ws[f'H{row}'] = justificacion
            wb.save(EXCEL_FILE)
            print(f"✅ Guardado  |  Prompt {w_prompt.value}  |  {w_modelo.value}")
            print(f"   Ranking: {ranking}")
            print(f"   Puntos:  A={punts['A']}  B={punts['B']}  C={punts['C']}  D={punts['D']}")
            w_texto.value = ""
            w_prompt.value = min(w_prompt.value + 1, 39)
        except PermissionError:
            print("❌ Cierra el Excel e intenta de nuevo")
        except Exception as e:
            print(f"❌ Error: {e}")

w_boton.on_click(guardar)

display(widgets.VBox([widgets.HBox([w_prompt, w_modelo]), w_texto, w_boton, w_estado]))